# ViRead VietOCR V2

This notebook resumes one of two profiles from the active, hash-checked VietOCR checkpoint, uploads a candidate in 40 MiB shards, and evaluates Original and V2 on a separately sealed Locked Test. It never trains PaddleOCR or Tesseract.

Create Colab Secrets `VIREAD_API_BASE` and `VIREAD_ADMIN_BEARER`. Register the training dataset and independent Locked Test in ViRead Admin. Keep the same Drive run folder, manifest, parent and settings when resuming.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, sys, os, json, hashlib, requests
drive.mount('/content/drive')
REPO=Path('/content/capstone1'); BRANCH='feature/viread-vietocr-v2-20261112'
if not (REPO/'ai/vietocr_v2/train.py').exists(): subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/trancuong3/capstone1.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'ai/vietocr_v2/requirements.txt')],check=True)
sys.path.insert(0,str(REPO)); os.chdir(REPO)
from ai.vietocr_v2.contracts import DatasetRow, ORIGINAL_CHECKPOINT_SHA256, write_jsonl, sha256_file
from ai.vietocr_v2.artifact_store import shard_file, assemble_shards
from ai.vietocr_v2.train import run_training
from ai.vietocr_v2.benchmark import run_locked_benchmark
API=userdata.get('VIREAD_API_BASE').rstrip('/'); TOKEN=userdata.get('VIREAD_ADMIN_BEARER')
if not API or not TOKEN: raise RuntimeError('Set VIREAD_API_BASE and VIREAD_ADMIN_BEARER in Colab Secrets.')
HEAD={'Authorization':f'Bearer {TOKEN}'}
def call(method,path,**kwargs):
 r=requests.request(method,API+path,headers=HEAD,timeout=300,**kwargs); r.raise_for_status(); return r.json() if r.content else {}
WORK=Path('/content/drive/MyDrive/ViRead/VietOCR-V2'); WORK.mkdir(parents=True,exist_ok=True)
print('Admin API connected; secret values are not displayed.')

## Pick data and profile
For profile A, leave the training version blank to snapshot Admin VERIFIED pages. For profile B, provide a registered Flow 2 version whose external labels and TRDG rows retain separate source metadata. No Locked Test sample is included in a training manifest.

In [ ]:
PROFILE='FLOW1_VERIFIED'  # or FLOW2_EXTERNAL_TRDG
TRAINING_VERSION_ID=''  # profile B requires a registered version ID
LOCKED_TEST_VERSION_ID=''
TOTAL_STEPS=5000; BATCH_SIZE=16; SEED=20261112; RESUME=True
if PROFILE=='FLOW1_VERIFIED' and not TRAINING_VERSION_ID:
 first=call('GET','/books/admin/viread-training/datasets/flow1',params={'limit':1,'offset':0}); TRAINING_VERSION_ID=first['dataset_version_id']
if not TRAINING_VERSION_ID or not LOCKED_TEST_VERSION_ID: raise RuntimeError('Set registered training and Locked Test version IDs.')
def fetch_dataset(version_id, flow1=False):
 if flow1: version_id=call('GET','/books/admin/viread-training/datasets/flow1',params={'limit':1,'offset':0})['dataset_version_id']
 first=call('GET',f'/books/admin/viread-training/datasets/{version_id}/samples',params={'limit':1,'offset':0}); dataset=first['dataset']
 receipt=requests.get(dataset['manifest_url'],timeout=120); receipt.raise_for_status(); receipt_path=WORK/f'{version_id}.receipt.json'; receipt_path.write_bytes(receipt.content)
 if sha256_file(receipt_path)!=dataset['version_sha256']: raise ValueError('Dataset receipt SHA-256 mismatch.')
 items=[]; offset=0
 while True:
  page=call('GET',f'/books/admin/viread-training/datasets/{version_id}/samples',params={'limit':200,'offset':offset}); items.extend(page['items'])
  if page['next_offset'] is None: break
  offset=page['next_offset']
 if not items: raise ValueError('Registered dataset version contains no samples.')
 rows=[]
 for item in items:
  image=WORK/'images'/item['image_sha256']; image.parent.mkdir(parents=True,exist_ok=True)
  if not image.exists():
   response=requests.get(item['image_url'],timeout=120); response.raise_for_status(); image.write_bytes(response.content)
  if sha256_file(image)!=item['image_sha256']: raise ValueError(f'Image hash mismatch: {item["sample_id"]}')
  flow=item.get('source_flow') or dataset['source_flow']
  if item['split']!='locked_test': rows.append(DatasetRow(str(image),item['label'],item['group_id'],flow,item['split'],item['image_sha256'],flow=='FLOW1_VERIFIED',item.get('source_annotation_id')))
 if not rows and dataset['source_flow']!='LOCKED_TEST': raise ValueError('Training dataset contains no eligible samples.')
 manifest=WORK/f'{version_id}.jsonl' if rows else None
 if rows: write_jsonl(manifest,rows)
 return dataset,receipt_path,manifest
training,TRAINING_RECEIPT,TRAINING_MANIFEST=fetch_dataset(TRAINING_VERSION_ID,PROFILE=='FLOW1_VERIFIED')
if training['is_locked_test']: raise ValueError('Locked Test cannot be used for training.')
print('Training samples:',len(TRAINING_MANIFEST.read_text(encoding='utf-8').splitlines()),'dataset SHA:',training['version_sha256'])

## Resume or train
The trainer stores model, optimizer, scheduler and random states to Drive every 100 steps. Candidate creation is strict: the upstream architecture, vocabulary and Original checkpoint hashes must match the repository lock.

In [ ]:
active=call('GET','/books/admin/viread-training/active-model/artifact')
PARENT=WORK/f'parent-{active["checkpoint_sha256"]}.pth'
if active['profile']=='ORIGINAL':
 if not PARENT.exists():
  response=requests.get(active['source_url'],stream=True,timeout=300); response.raise_for_status()
  with PARENT.open('wb') as out:
   for block in response.iter_content(1024*1024):
    if block: out.write(block)
else:
 shard_dir=WORK/f'parent-shards-{active["checkpoint_sha256"]}'; shard_dir.mkdir(parents=True,exist_ok=True); parts=[]
 if not PARENT.exists():
  for part in active['chunks']:
   local=shard_dir/f'part-{int(part["index"]):04d}.bin'
   if not local.exists():
    response=requests.get(part['url'],timeout=300); response.raise_for_status(); local.write_bytes(response.content)
   if sha256_file(local)!=part['sha256']: raise ValueError('Parent shard hash mismatch.')
   parts.append({'index':part['index'],'name':local.name,'bytes':part['bytes'],'sha256':part['sha256']})
  assemble_shards({'schema':'viread-checkpoint-shards-v1','filename':'parent.pth','bytes':sum(p['bytes'] for p in parts),'sha256':active['checkpoint_sha256'],'chunks':parts},shard_dir,PARENT)
if sha256_file(PARENT)!=active['checkpoint_sha256']: raise ValueError('Active parent checkpoint hash mismatch.')
RUN=WORK/f'{PROFILE}-{training["version_sha256"][:12]}'; RUN.mkdir(parents=True,exist_ok=True)
candidate=run_training(profile=PROFILE,manifest_path=str(TRAINING_MANIFEST),output_dir=str(RUN),parent_checkpoint=str(PARENT),
 total_steps=TOTAL_STEPS,batch_size=BATCH_SIZE,seed=SEED,resume=RESUME,
 dataset_receipt_path=str(TRAINING_RECEIPT),dataset_version_sha256=training['version_sha256'])
print(json.dumps({k:v for k,v in candidate.items() if k!='candidate_path'},indent=2))

## Upload candidate and run the paired benchmark
The backend reassembles and verifies every shard. It recomputes metrics from candidate outputs and the private human references. A passing gate enables a separate Admin approval action; this notebook does not promote a model.

In [ ]:
run=call('POST','/books/admin/viread-training/runs',json={'profile':PROFILE,'dataset_version_id':TRAINING_VERSION_ID,
 'parent_checkpoint_sha256':candidate['parent_checkpoint_sha256'],'manifest_sha256':training['version_sha256'],'config_sha256':candidate['config_sha256'],
 'architecture_sha256':candidate['architecture_sha256'],'vocabulary_sha256':candidate['vocabulary_sha256']})
RUN_ID=run['run_id']; chunks_dir=RUN/'upload-shards'; shards=shard_file(candidate['candidate_path'],chunks_dir)
for part in shards['chunks']:
 with (chunks_dir/part['name']).open('rb') as stream:
  call('POST',f'/books/admin/viread-training/runs/{RUN_ID}/chunks',data={'chunk_index':str(part['index'])},files={'file':(part['name'],stream,'application/octet-stream')})
call('POST',f'/books/admin/viread-training/runs/{RUN_ID}/complete',json={'profile':candidate['profile'],'parent_checkpoint_sha256':candidate['parent_checkpoint_sha256'],
 'manifest_sha256':candidate['manifest_sha256'],'config_sha256':candidate['config_sha256'],'architecture_sha256':candidate['architecture_sha256'],
 'vocabulary_sha256':candidate['vocabulary_sha256'],'checkpoint_sha256':candidate['checkpoint_sha256'],
 'checkpoint_bytes':Path(candidate['candidate_path']).stat().st_size,'step':candidate['steps']})
locked,LOCKED_RECEIPT,_=fetch_dataset(LOCKED_TEST_VERSION_ID)
if not locked['is_locked_test']: raise ValueError('Selected test dataset is not sealed Locked Test.')
original=WORK/f'original-{ORIGINAL_CHECKPOINT_SHA256}.pth'
if not original.exists():
 response=requests.get('https://vocr.vn/data/vietocr/vgg_transformer.pth',stream=True,timeout=300); response.raise_for_status()
 with original.open('wb') as out:
  for block in response.iter_content(1024*1024):
   if block: out.write(block)
if sha256_file(original)!=ORIGINAL_CHECKPOINT_SHA256: raise ValueError('Original checkpoint hash mismatch.')
report=RUN/'locked-test-report.json'
result=run_locked_benchmark(manifest_path=str(LOCKED_RECEIPT),expected_manifest_sha256=locked['version_sha256'],
 original_checkpoint=str(original),candidate_checkpoint=candidate['candidate_path'],output_path=str(report),image_cache_dir=str(WORK/'images'))
if result['candidate_checkpoint_sha256']!=candidate['checkpoint_sha256']: raise ValueError('Benchmarked candidate differs from the uploaded checkpoint.')
outputs=[{k:item[k] for k in ('sample_id','group_id','original','candidate')} for item in result['outputs']]
registered=call('POST','/books/admin/viread-training/benchmarks',json={'run_id':RUN_ID,'locked_test_version_id':LOCKED_TEST_VERSION_ID,
 'locked_test_manifest_sha256':locked['version_sha256'],'outputs':outputs})
print(json.dumps(registered,ensure_ascii=False,indent=2))
print('Review the report in Admin. A human approval is still required.')